In [ ]:
# import libraries
try:
  # %tensorflow_version only exists in Colab.
  !pip install tf-nightly
except Exception:
  pass
import tensorflow as tf
import pandas as pd
from tensorflow import keras
!pip install tensorflow-datasets
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
# Load the dataset into pandas DataFrame
train_data = pd.read_csv(train_file_path, sep='\t', header=None, names=['label', 'message'])
test_data = pd.read_csv(test_file_path, sep='\t', header=None, names=['label', 'message'])

# Check the data
train_data.head(), test_data.head()

# Map labels to numbers: "ham" -> 0, "spam" -> 1
train_data['label'] = train_data['label'].map({'ham': 0, 'spam': 1})
test_data['label'] = test_data['label'].map({'ham': 0, 'spam': 1})

# Tokenization and padding of text data
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Tokenizer for converting text into integer tokens
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(train_data['message'])

X_train = tokenizer.texts_to_sequences(train_data['message'])
X_test = tokenizer.texts_to_sequences(test_data['message'])

# Padding sequences to make them of equal length
max_len = 150  # Maximum length of sequence
X_train_pad = pad_sequences(X_train, padding='post', maxlen=max_len)
X_test_pad = pad_sequences(X_test, padding='post', maxlen=max_len)

# Labels
y_train = train_data['label'].values
y_test = test_data['label'].values


In [ ]:
# Build the model
model = keras.Sequential([
    layers.Embedding(input_dim=5000, output_dim=64, input_length=max_len),
    layers.GlobalAveragePooling1D(),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')  # Sigmoid for binary classification
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(X_train_pad, y_train, epochs=5, batch_size=32, validation_data=(X_test_pad, y_test))


In [ ]:
# Function to predict messages
def predict_message(pred_text):
    # Tokenize and pad the input text
    seq = tokenizer.texts_to_sequences([pred_text])
    padded = pad_sequences(seq, padding='post', maxlen=max_len)

    # Predict the probability of being spam
    prob_spam = model.predict(padded)[0][0]
    label = 'spam' if prob_spam > 0.5 else 'ham'

    return [prob_spam, label]

# Test the function with an example message
pred_text = "how are you doing today?"
prediction = predict_message(pred_text)
print(prediction)  # Output should be something like [0.0083, 'ham']


In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
